# controlled_unlearn_beta005_preserve_001

Notebook Kaggle reproductible préparé en mode **no-submit strict**.

Objectif: partir du checkpoint champion `interp20_30_beta0p05`, charger un student et un teacher frozen identiques, puis préparer un unlearning contrôlé ciblé poison avec préservation forte hors poison.

IMPORTANT:
- Ce notebook ne soumet rien à Kaggle.
- `RUN_TRAINING = False` par défaut: aucune boucle d'entraînement ne s'exécute sans modification humaine explicite.
- `DRY_RUN_PREFLIGHT_ONLY = True` par défaut: le notebook vérifie les chemins/garde-fous et imprime le coût estimé, les fichiers prévus et les conditions d'arrêt.
- Toute exécution future avec entraînement doit rester no-submit et nécessite validation humaine séparée.


In [ ]:
# =============================
# 0. Guardrails immuables
# =============================
RUN_ID = "controlled_unlearn_beta005_preserve_001"
MODE = "no_submit_prepare_only_by_default"
RUN_TRAINING = False          # NE PAS changer sans accord humain explicite de lancement no-submit
DRY_RUN_PREFLIGHT_ONLY = True # doit rester True tant que le run n'est pas explicitement autorisé
ALLOW_KAGGLE_SUBMIT = False
AUTO_SUBMIT = False
SUBMISSION_AUTHORIZED = False
PAID_GPU_AUTHORIZED = False
ACTIVE_RUN_EXPECTED = None

assert ALLOW_KAGGLE_SUBMIT is False
assert AUTO_SUBMIT is False
assert SUBMISSION_AUTHORIZED is False
assert PAID_GPU_AUTHORIZED is False
print({
    "run_id": RUN_ID,
    "mode": MODE,
    "run_training": RUN_TRAINING,
    "dry_run_preflight_only": DRY_RUN_PREFLIGHT_ONLY,
    "auto_submit": AUTO_SUBMIT,
    "submission_authorized": SUBMISSION_AUTHORIZED,
    "paid_gpu_authorized": PAID_GPU_AUTHORIZED,
    "active_run_expected": ACTIVE_RUN_EXPECTED,
})


## Coût estimé, fichiers créés, conditions d'arrêt

Coût estimé si exécution future autorisée: environ `~6h` GPU Kaggle T4 gratuit/no-submit.
Décomposition prévue: setup/preflight `~0.5h`, entraînement B1 `~1.5–2h`, inférence/audits full-test `~3–4h`.
Aucun GPU payant n'est autorisé par ce notebook.

Sorties prévues sous `/kaggle/working/artifacts/controlled_unlearn_beta005_preserve_001/`:
- `config/frozen_protocol.json`
- `audits/preflight_guardrails.json`
- `audits/preflight_files.json`
- `audits/preflight_scope_audit.json`
- `logs/train_log.jsonl`
- checkpoints B1: `checkpoints/B1_lr1e-6_iter50.pth`, `B1_lr1e-6_iter100.pth`, `B1_lr2e-6_iter50.pth`, `B1_lr2e-6_iter100.pth`
- CSV/audits pour checkpoints évalués: `csv/*.csv`, `audits/*_full_audit.json`
- `candidate_ranking.csv`
- `top_submit_review.json`
- `sha256_manifest.json`
- `reports/controlled_unlearn_beta005_preserve_001_no_submit_report.md`

Conditions d'arrêt hard reject:
- garde-fous non vérifiés: `active_run != null`, `auto_submit != false`, `submission_authorized != false`, `paid_gpu_authorized != false`
- checkpoint beta0p05 absent ou SHA256 différent de `eff02bca7db558ba1fa31da6a46406009f77dde4be2c5c91b4a6451512762320`
- Detectron2/config/ancres/preprocessing incompatibles
- trainables autres que `head.cls_subnet.6.*` et `head.cls_score.*`
- modification d'un paramètre gelé
- CSV invalide, rows != 2000, valeurs non finies, prediction_string mal groupée
- detections `<1700`, empty images `>880`, confidence_sum `<700`
- recall vs beta0p05 `<0.955`, precision vs beta0p05 `<0.965`, CPSR red
- effet inférieur au bruit expérimental ou non localisé poison
- présence d'une action leaderboard écriture


In [ ]:
# =============================
# 1. Imports, chemins, protocole
# =============================
import os, sys, json, math, time, hashlib, random, subprocess, shutil
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

try:
    import torch
    import torch.nn.functional as F
except Exception as e:
    raise RuntimeError("PyTorch requis") from e

# Chemins Kaggle attendus. Ajuster uniquement les chemins d'input si le dataset Kaggle custom porte un autre nom.
COMPETITION_DIR = Path(os.environ.get("COMPETITION_DIR", "/kaggle/input/neural-debris-removal-in-streak-detection-models"))
BETA005_INPUT_DIR = Path(os.environ.get("BETA005_INPUT_DIR", "/kaggle/input/neural-beta005-reference"))
BETA005_CKPT = Path(os.environ.get("BETA005_CKPT", str(BETA005_INPUT_DIR / "interp20_30_beta0p05_reference.pth")))
BETA005_REF_CSV = Path(os.environ.get("BETA005_REF_CSV", str(BETA005_INPUT_DIR / "interp20_30_beta0p05_reference.csv")))
RUNTIME_STATE_PATH = Path(os.environ.get("RUNTIME_STATE_PATH", "/kaggle/input/neural-runtime-state/runtime_state.json"))

UNLEARN_DIR = COMPETITION_DIR / "unlearn_set"
ANNOTATIONS_JSON = UNLEARN_DIR / "annotations_coco.json"
TEST_DIR = COMPETITION_DIR / "test_set"
SAMPLE_SUBMISSION = COMPETITION_DIR / "sample_submission.csv"
POISONED_CKPT = COMPETITION_DIR / "poisoned_model.pth"

OUTPUT_ROOT = Path("/kaggle/working/artifacts") / RUN_ID
for sub in ["config", "audits", "logs", "checkpoints", "csv", "reports"]:
    (OUTPUT_ROOT / sub).mkdir(parents=True, exist_ok=True)

EXPECTED_BETA005_SHA256 = "eff02bca7db558ba1fa31da6a46406009f77dde4be2c5c91b4a6451512762320"
EXPECTED_POISONED_SHA256 = "f6c21faa2a5b56549fc9e058147c90b149a034858fe0678f5a99ea5a6f0e657c"
EXPECTED_REF = {
    "detections_total": 1831,
    "empty_images": 806,
    "confidence_sum": 750.4149846,
    "cpsr_zone": "green",
}

PROTOCOL = {
    "run_id": RUN_ID,
    "mode": MODE,
    "competition_slug": "neural-debris-removal-in-streak-detection-models",
    "active_reference": "interp20_30_beta0p05",
    "active_reference_public_score": 257.6416,
    "metric_direction": "lower_is_better",
    "student_init": str(BETA005_CKPT),
    "teacher_frozen": str(BETA005_CKPT),
    "expected_beta005_sha256": EXPECTED_BETA005_SHA256,
    "detectron2_config": "COCO-Detection/retinanet_R_50_FPN_3x.yaml",
    "anchor_ratios": [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0],
    "anchor_sizes": [[16], [32], [64], [128], [256]],
    "score_threshold": 0.2,
    "preprocessing": "preserve I;16/raw uint16 PNG, scale to 0..255, duplicate 3 channels; do not PIL.convert('RGB') directly",
    "trainable_scope_B1": ["head.cls_subnet.6.weight", "head.cls_subnet.6.bias", "head.cls_score.weight", "head.cls_score.bias"],
    "frozen": ["backbone", "FPN", "bbox head", "head.cls_subnet.0.*", "head.cls_subnet.2.*", "head.cls_subnet.4.*"],
    "loss_weights": {"negative_poison": 1.0, "preserve_outside": 8.0, "l2sp": 30.0, "rank_preserve": 2.0, "corridor": 3.0},
    "sweep_prepared_not_executed_by_default": [
        {"name": "B1_lr1e-6_iter50", "lr": 1e-6, "iter": 50},
        {"name": "B1_lr1e-6_iter100", "lr": 1e-6, "iter": 100},
        {"name": "B1_lr2e-6_iter50", "lr": 2e-6, "iter": 50},
        {"name": "B1_lr2e-6_iter100", "lr": 2e-6, "iter": 100},
    ],
    "submit_review_corridor": {"detections": [1750,1885], "empty_images": [775,850], "confidence_sum": [710,765], "precision_vs_beta0p05_min": 0.975, "recall_vs_beta0p05_min": 0.970, "cpsr": "green_or_low_orange"},
    "hard_reject": {"detections_lt":1700, "empty_gt":880, "confidence_sum_lt":700, "recall_lt":0.955, "precision_lt":0.965, "cpsr":"red"},
    "decision_values": ["reject", "rerun", "demander accord humain pour submit"],
}
(OUTPUT_ROOT / "config" / "frozen_protocol.json").write_text(json.dumps(PROTOCOL, indent=2), encoding="utf-8")
print("Output root:", OUTPUT_ROOT)
print(json.dumps(PROTOCOL, indent=2)[:4000])


In [ ]:
# =============================
# 2. Fonctions utilitaires: SHA, preflight, interdiction submit
# =============================
def sha256_file(path: Path, block_size: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(block_size), b""):
            h.update(block)
    return h.hexdigest()

def write_json(path: Path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, sort_keys=True), encoding="utf-8")

def assert_no_submit_code():
    # Défense volontaire: ne jamais appeler une action leaderboard dans ce notebook.
    forbidden = ["leaderboard_write_action"]
    nb_text = ""
    try:
        import __main__
        nb_text = " ".join(str(v) for v in globals().values() if isinstance(v, str))
    except Exception:
        pass
    # Cette vérification est indicative en runtime; la garantie principale est qu'aucune fonction de submit n'existe.
    assert ALLOW_KAGGLE_SUBMIT is False
    return {"allow_kaggle_submit": ALLOW_KAGGLE_SUBMIT, "submit_function_defined": False, "forbidden_api_intent": False}

def preflight_guardrails():
    state = None
    if RUNTIME_STATE_PATH.exists():
        try:
            state = json.loads(RUNTIME_STATE_PATH.read_text())
        except Exception as e:
            raise RuntimeError(f"runtime_state illisible: {RUNTIME_STATE_PATH}: {e}")
        assert state.get("active_run", None) is None, state.get("active_run")
        assert state.get("auto_submit", None) is False, state.get("auto_submit")
        assert state.get("submission_authorized", None) is False, state.get("submission_authorized")
        assert state.get("paid_gpu_authorized", None) is False, state.get("paid_gpu_authorized")
    else:
        # Kaggle peut ne pas avoir runtime_state local; on vérifie les constantes notebook strictes.
        state = {"note": "runtime_state.json absent dans les inputs Kaggle; constantes notebook utilisées"}
    assert ACTIVE_RUN_EXPECTED is None
    assert AUTO_SUBMIT is False
    assert SUBMISSION_AUTHORIZED is False
    assert PAID_GPU_AUTHORIZED is False
    out = {
        "ok": True,
        "runtime_state_path": str(RUNTIME_STATE_PATH),
        "runtime_state_present": RUNTIME_STATE_PATH.exists(),
        "active_run": None,
        "auto_submit": AUTO_SUBMIT,
        "submission_authorized": SUBMISSION_AUTHORIZED,
        "paid_gpu_authorized": PAID_GPU_AUTHORIZED,
        "run_training": RUN_TRAINING,
        "dry_run_preflight_only": DRY_RUN_PREFLIGHT_ONLY,
        "checked_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    write_json(OUTPUT_ROOT / "audits" / "preflight_guardrails.json", out)
    return out

def preflight_files():
    required = {
        "competition_dir": COMPETITION_DIR,
        "unlearn_dir": UNLEARN_DIR,
        "annotations_json": ANNOTATIONS_JSON,
        "test_dir": TEST_DIR,
        "sample_submission": SAMPLE_SUBMISSION,
        "beta005_checkpoint": BETA005_CKPT,
        "beta005_reference_csv": BETA005_REF_CSV,
    }
    report = {k: {"path": str(p), "exists": p.exists()} for k,p in required.items()}
    missing = [k for k,v in report.items() if not v["exists"]]
    if missing:
        report["ok"] = False
        report["missing"] = missing
        write_json(OUTPUT_ROOT / "audits" / "preflight_files.json", report)
        raise FileNotFoundError(f"Fichiers manquants: {missing}. Ajouter les datasets Kaggle requis avant exécution future.")
    report["beta005_sha256"] = sha256_file(BETA005_CKPT)
    report["beta005_sha256_ok"] = report["beta005_sha256"] == EXPECTED_BETA005_SHA256
    assert report["beta005_sha256_ok"], report["beta005_sha256"]
    if POISONED_CKPT.exists():
        report["poisoned_sha256"] = sha256_file(POISONED_CKPT)
        report["poisoned_sha256_ok"] = report["poisoned_sha256"] == EXPECTED_POISONED_SHA256
    sample = pd.read_csv(SAMPLE_SUBMISSION)
    report["sample_rows"] = int(len(sample))
    report["sample_columns"] = list(sample.columns)
    assert len(sample) == 2000
    assert "prediction_string" in sample.columns
    report["ok"] = True
    write_json(OUTPUT_ROOT / "audits" / "preflight_files.json", report)
    return report

print("Preflight guardrails:", json.dumps(preflight_guardrails(), indent=2))
print("No-submit code guard:", json.dumps(assert_no_submit_code(), indent=2))
# preflight_files() est volontairement appelable; en environnement de préparation local sans inputs Kaggle, il peut échouer.


In [ ]:
# =============================
# 3. Install/import Detectron2 (cellule future)
# =============================
# Exécuter cette cellule sur Kaggle seulement si detectron2 n'est pas déjà disponible.
# Elle ne soumet rien; elle prépare l'environnement officiel Detectron2.
try:
    import detectron2
    print("detectron2 already available", detectron2.__version__ if hasattr(detectron2, "__version__") else "unknown")
except Exception:
    if not RUN_TRAINING:
        print("detectron2 absent; installation non lancée car RUN_TRAINING=False / preflight-only.")
    else:
        # Reprend la convention des notebooks publics validés: setuptools<81 puis detectron2 depuis GitHub.
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "setuptools<81"])
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/facebookresearch/detectron2.git"])
        import detectron2
        print("detectron2 installed")


In [ ]:
# =============================
# 4. Construction modèle officiel RetinaNet beta0p05
# =============================
def build_cfg(checkpoint_path: Path, score_thresh: float = 0.2):
    from detectron2 import model_zoo
    from detectron2.config import get_cfg
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file("COCO-Detection/retinanet_R_50_FPN_3x.yaml"))
    cfg.MODEL.RETINANET.NUM_CLASSES = 1
    cfg.MODEL.ANCHOR_GENERATOR.ASPECT_RATIOS = [[0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]]
    cfg.MODEL.ANCHOR_GENERATOR.SIZES = [[16], [32], [64], [128], [256]]
    cfg.MODEL.RETINANET.SCORE_THRESH_TEST = score_thresh
    cfg.MODEL.WEIGHTS = str(checkpoint_path)
    cfg.INPUT.MIN_SIZE_TEST = 1024
    cfg.INPUT.MAX_SIZE_TEST = 1024
    cfg.DATALOADER.NUM_WORKERS = 2
    cfg.SOLVER.IMS_PER_BATCH = 2
    return cfg

def load_model_from_beta005(device="cuda"):
    from detectron2.modeling import build_model
    from detectron2.checkpoint import DetectionCheckpointer
    cfg = build_cfg(BETA005_CKPT, score_thresh=0.2)
    student = build_model(cfg)
    teacher = build_model(cfg)
    DetectionCheckpointer(student).load(str(BETA005_CKPT))
    DetectionCheckpointer(teacher).load(str(BETA005_CKPT))
    student.to(device); teacher.to(device)
    teacher.eval()
    for p in teacher.parameters():
        p.requires_grad = False
    return cfg, student, teacher

TRAINABLE_NAMES_B1 = {
    "head.cls_subnet.6.weight",
    "head.cls_subnet.6.bias",
    "head.cls_score.weight",
    "head.cls_score.bias",
}
EXPECTED_TRAINABLE_PARAM_COUNT_B1 = 606215

def apply_freeze_b1(model):
    trainable = []
    frozen = []
    total_trainable = 0
    for name, p in model.named_parameters():
        if name in TRAINABLE_NAMES_B1:
            p.requires_grad = True
            trainable.append(name)
            total_trainable += p.numel()
        else:
            p.requires_grad = False
            frozen.append(name)
    assert set(trainable) == TRAINABLE_NAMES_B1, trainable
    assert total_trainable == EXPECTED_TRAINABLE_PARAM_COUNT_B1, total_trainable
    forbidden_trainable = [n for n,p in model.named_parameters() if p.requires_grad and n not in TRAINABLE_NAMES_B1]
    assert not forbidden_trainable, forbidden_trainable
    audit = {
        "ok": True,
        "trainable_names": trainable,
        "trainable_param_count": total_trainable,
        "expected_trainable_param_count": EXPECTED_TRAINABLE_PARAM_COUNT_B1,
        "frozen_param_tensors": len(frozen),
        "freeze_backbone_fpn_bbox_early_cls": True,
    }
    write_json(OUTPUT_ROOT / "audits" / "preflight_scope_audit.json", audit)
    return audit

print("Scope B1 attendu:", sorted(TRAINABLE_NAMES_B1), "params", EXPECTED_TRAINABLE_PARAM_COUNT_B1)


In [ ]:
# =============================
# 5. Loader 16-bit + zones poison/preserve
# =============================
import cv2

def read_science_png_as_bgr255(path: Path) -> np.ndarray:
    raw = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if raw is None:
        raise FileNotFoundError(path)
    if raw.ndim == 3:
        raw = raw[..., 0]
    if raw.dtype == np.uint16:
        img = (raw.astype(np.float32) / 65535.0) * 255.0
    else:
        arr = raw.astype(np.float32)
        mx = float(arr.max()) if arr.size else 0.0
        img = arr if mx <= 255.0 else (arr / mx) * 255.0
    img = np.clip(img, 0, 255).astype(np.uint8)
    return np.repeat(img[:, :, None], 3, axis=2)  # Detectron2 attend BGR; grayscale répété => BGR=RGB

def load_unlearn_annotations():
    data = json.loads(ANNOTATIONS_JSON.read_text())
    images = {im["id"]: im for im in data["images"]}
    anns_by_image = {im_id: [] for im_id in images}
    for ann in data.get("annotations", []):
        anns_by_image.setdefault(ann["image_id"], []).append(ann)
    return images, anns_by_image

def coco_xywh_to_xyxy(b):
    x, y, w, h = map(float, b)
    return np.array([x, y, x+w, y+h], dtype=np.float32)

def expand_box_xyxy(box, factor=1.25, min_extra=0, width=1024, height=1024):
    x1,y1,x2,y2 = map(float, box)
    cx, cy = (x1+x2)/2, (y1+y2)/2
    bw, bh = (x2-x1), (y2-y1)
    new_w = max(bw*factor, bw + min_extra)
    new_h = max(bh*factor, bh + min_extra)
    out = np.array([cx-new_w/2, cy-new_h/2, cx+new_w/2, cy+new_h/2], dtype=np.float32)
    out[[0,2]] = np.clip(out[[0,2]], 0, width)
    out[[1,3]] = np.clip(out[[1,3]], 0, height)
    return out

def point_in_box(xy, box):
    x, y = xy
    x1,y1,x2,y2 = box
    return (x >= x1) and (x <= x2) and (y >= y1) and (y <= y2)

print("Loader 16-bit prêt; ne pas utiliser PIL.convert('RGB') sur les PNG I;16.")


In [ ]:
# =============================
# 6. Losses préparées: unlearn / preserve / l2sp / rank / corridor
# =============================
def sigmoid_logits_from_head_outputs(head_outputs):
    # RetinaNet head logits: list[level] tensors [N, A*K, H, W]. NUM_CLASSES=1 donc A*K=A=7.
    return [torch.sigmoid(x) for x in head_outputs]

def mse_masked(student_logits, teacher_logits, masks, weights=None):
    vals = []
    for s, t, m in zip(student_logits, teacher_logits, masks):
        # m shape broadcastable [N, A, H, W]
        if m.sum() == 0:
            continue
        diff2 = (s - t).pow(2)
        if weights is not None:
            vals.append((diff2 * m * weights).sum() / (m.sum() + 1e-6))
        else:
            vals.append((diff2 * m).sum() / (m.sum() + 1e-6))
    if not vals:
        return torch.tensor(0.0, device=student_logits[0].device)
    return torch.stack(vals).mean()

def loss_l2sp(model, anchor_state, trainable_names=TRAINABLE_NAMES_B1):
    vals = []
    for name, p in model.named_parameters():
        if name in trainable_names:
            vals.append((p - anchor_state[name].to(p.device)).pow(2).mean())
    return torch.stack(vals).mean() if vals else torch.tensor(0.0)

def loss_negative_poison(student_logits, teacher_logits, target_masks, min_teacher_prob=0.05, soft_min=0.05, soft_max=0.15, reduce_factor=0.70):
    vals = []
    for s_logit, t_logit, m in zip(student_logits, teacher_logits, target_masks):
        with torch.no_grad():
            pt = torch.sigmoid(t_logit)
            active = (m > 0) & (pt >= min_teacher_prob)
            target = torch.clamp(pt * reduce_factor, min=soft_min, max=soft_max)
        if active.sum() == 0:
            continue
        vals.append(F.binary_cross_entropy_with_logits(s_logit[active], target[active], reduction="mean"))
    if not vals:
        return torch.tensor(0.0, device=student_logits[0].device)
    return torch.stack(vals).mean()

def loss_preserve_outside(student_logits, teacher_logits, preserve_masks):
    vals = []
    for s, t, m in zip(student_logits, teacher_logits, preserve_masks):
        with torch.no_grad():
            w = torch.where(torch.sigmoid(t) >= 0.20, torch.tensor(3.0, device=t.device), torch.tensor(1.0, device=t.device))
        denom = m.sum() + 1e-6
        vals.append(((s - t).pow(2) * m * w).sum() / denom)
    return torch.stack(vals).mean() if vals else torch.tensor(0.0, device=student_logits[0].device)

def loss_corridor(student_logits, teacher_logits, preserve_masks, lo=0.96, hi=1.01):
    vals = []
    for s, t, m in zip(student_logits, teacher_logits, preserve_masks):
        denom = m.sum() + 1e-6
        ms = (torch.sigmoid(s) * m).sum() / denom
        mt = (torch.sigmoid(t) * m).sum() / denom
        ratio = ms / (mt + 1e-8)
        vals.append(torch.relu(torch.tensor(lo, device=s.device) - ratio).pow(2) + torch.relu(ratio - torch.tensor(hi, device=s.device)).pow(2))
    return torch.stack(vals).mean() if vals else torch.tensor(0.0, device=student_logits[0].device)

def loss_rank_preserve(student_logits, teacher_logits, preserve_masks, topk=200, margin=0.01):
    # Pairwise léger: préserver que les réponses teacher top-k restent hautes relativement aux réponses autour.
    vals = []
    for s, t, m in zip(student_logits, teacher_logits, preserve_masks):
        flat_m = m.flatten() > 0
        if flat_m.sum() < topk + 2:
            continue
        s_flat = s.flatten()[flat_m]
        t_flat = t.flatten()[flat_m]
        k = min(topk, t_flat.numel())
        idx = torch.topk(t_flat.detach(), k=k).indices
        # MSE sur logits top-k comme proxy ranking stable; évite O(k^2) coûteux.
        vals.append(F.mse_loss(s_flat[idx], t_flat.detach()[idx]))
    return torch.stack(vals).mean() if vals else torch.tensor(0.0, device=student_logits[0].device)

def total_controlled_unlearn_loss(student_logits, teacher_logits, target_masks, preserve_masks, model, anchor_state):
    l_neg = loss_negative_poison(student_logits, teacher_logits, target_masks)
    l_pres = loss_preserve_outside(student_logits, teacher_logits, preserve_masks)
    l_l2 = loss_l2sp(model, anchor_state)
    l_rank = loss_rank_preserve(student_logits, teacher_logits, preserve_masks)
    l_corr = loss_corridor(student_logits, teacher_logits, preserve_masks)
    total = 1.0*l_neg + 8.0*l_pres + 30.0*l_l2 + 2.0*l_rank + 3.0*l_corr
    return total, {"negative_poison": float(l_neg.detach().cpu()), "preserve_outside": float(l_pres.detach().cpu()), "l2sp": float(l_l2.detach().cpu()), "rank_preserve": float(l_rank.detach().cpu()), "corridor": float(l_corr.detach().cpu())}

print("Losses préparées: negative_poison, preserve_outside, l2sp, rank_preserve, corridor.")


In [ ]:
# =============================
# 7. Sweep préparé mais non exécuté par défaut
# =============================
SWEEP_B1 = PROTOCOL["sweep_prepared_not_executed_by_default"]

def run_training_sweep_prepared_only():
    if not RUN_TRAINING or DRY_RUN_PREFLIGHT_ONLY:
        print("Aucun entraînement lancé: RUN_TRAINING=False ou DRY_RUN_PREFLIGHT_ONLY=True.")
        print("Sweep préparé:")
        for item in SWEEP_B1:
            print(item)
        return {"training_started": False, "sweep": SWEEP_B1}
    raise RuntimeError("Boucle d'entraînement non lancée dans cette version préparatoire. Remplacer par implémentation autorisée après accord humain explicite.")

sweep_status = run_training_sweep_prepared_only()
write_json(OUTPUT_ROOT / "audits" / "sweep_prepare_status.json", sweep_status)


In [ ]:
# =============================
# 8. Audits finaux prévus: CSV, stats, precision/recall vs beta0p05, CPSR
# =============================
def parse_prediction_string(ps):
    if not isinstance(ps, str) or not ps.strip():
        return []
    vals = [float(x) for x in ps.split()]
    if len(vals) % 5 != 0:
        raise ValueError("prediction_string not groups of 5")
    return [vals[i:i+5] for i in range(0, len(vals), 5)]

def audit_submission_csv(csv_path: Path):
    df = pd.read_csv(csv_path)
    assert len(df) == 2000, len(df)
    assert "prediction_string" in df.columns
    dets = 0; empty = 0; conf_sum = 0.0; invalid = 0; nonfinite = 0
    for ps in df["prediction_string"].fillna("").astype(str):
        try:
            boxes = parse_prediction_string(ps)
        except Exception:
            invalid += 1; continue
        if not boxes:
            empty += 1
        dets += len(boxes)
        for b in boxes:
            if not all(math.isfinite(x) for x in b):
                nonfinite += 1
            conf_sum += b[0]
    return {"rows": int(len(df)), "detections_total": int(dets), "empty_images": int(empty), "confidence_sum": float(conf_sum), "invalid_prediction_strings": invalid, "nonfinite_values": nonfinite}

def iou_xyxy(a, b):
    ax1,ay1,ax2,ay2 = a; bx1,by1,bx2,by2 = b
    ix1,iy1 = max(ax1,bx1), max(ay1,by1)
    ix2,iy2 = min(ax2,bx2), min(ay2,by2)
    iw,ih = max(0.0, ix2-ix1), max(0.0, iy2-iy1)
    inter = iw*ih
    area_a=max(0,ax2-ax1)*max(0,ay2-ay1); area_b=max(0,bx2-bx1)*max(0,by2-by1)
    return inter/(area_a+area_b-inter+1e-9)

def compare_csv_vs_reference(candidate_csv: Path, ref_csv: Path = BETA005_REF_CSV, iou_thr=0.5):
    cand = pd.read_csv(candidate_csv)
    ref = pd.read_csv(ref_csv)
    key = "image_id" if "image_id" in cand.columns else cand.columns[0]
    ref_key = "image_id" if "image_id" in ref.columns else ref.columns[0]
    cdict = dict(zip(cand[key], cand["prediction_string"].fillna("").astype(str)))
    rdict = dict(zip(ref[ref_key], ref["prediction_string"].fillna("").astype(str)))
    matched_ref = 0; matched_cand = 0; total_ref = 0; total_cand = 0
    for image_id, rps in rdict.items():
        rboxes = parse_prediction_string(rps)
        cboxes = parse_prediction_string(cdict.get(image_id, ""))
        total_ref += len(rboxes); total_cand += len(cboxes)
        used_c = set()
        for rb in rboxes:
            rxy = rb[1:5]
            best_j, best_iou = None, 0.0
            for j, cb in enumerate(cboxes):
                if j in used_c: continue
                val = iou_xyxy(rxy, cb[1:5])
                if val > best_iou:
                    best_iou, best_j = val, j
            if best_iou >= iou_thr:
                matched_ref += 1
                used_c.add(best_j)
        matched_cand += len(used_c)
    return {
        "reference_detections": total_ref,
        "candidate_detections": total_cand,
        "matched_iou_ge_0p5": matched_ref,
        "recall_vs_beta0p05": matched_ref / max(1, total_ref),
        "precision_vs_beta0p05": matched_cand / max(1, total_cand),
        "dropped_beta0p05": total_ref - matched_ref,
        "new_vs_beta0p05": total_cand - matched_cand,
    }

def compute_cpsr_basic(global_stats, comp):
    # Conservative project gate approximation; full CPSR can be substituted by project implementation if available.
    flags = []
    if global_stats["detections_total"] < 1700: flags.append("detections_lt_1700")
    if global_stats["empty_images"] > 880: flags.append("empty_gt_880")
    if global_stats["confidence_sum"] < 700: flags.append("confidence_sum_lt_700")
    if comp["recall_vs_beta0p05"] < 0.955: flags.append("recall_lt_0p955")
    if comp["precision_vs_beta0p05"] < 0.965: flags.append("precision_lt_0p965")
    zone = "red" if flags else "green"
    return {"zone": zone, "hard_reject_flags": flags, "risk_score": 100 if flags else 0}

def final_audit_for_candidate(candidate_name, csv_path: Path):
    stats = audit_submission_csv(csv_path)
    comp = compare_csv_vs_reference(csv_path)
    cpsr = compute_cpsr_basic(stats, comp)
    audit = {"candidate": candidate_name, "csv": str(csv_path), "global_stats": stats, "comparison_vs_beta0p05": comp, "cpsr": cpsr}
    write_json(OUTPUT_ROOT / "audits" / f"{candidate_name}_full_audit.json", audit)
    return audit

print("Audits prévus: detections, empty_images, confidence_sum, precision/recall vs beta0p05, CPSR.")


In [ ]:
# =============================
# 9. Décision no-submit: reject / rerun / demander accord humain pour submit
# =============================
def decide_no_submit(audit, poison_effect_ratio=None, outside_drift_ok=None):
    stats = audit["global_stats"]
    comp = audit["comparison_vs_beta0p05"]
    cpsr = audit["cpsr"]
    hard = list(cpsr.get("hard_reject_flags", []))
    if stats.get("invalid_prediction_strings", 1) != 0 or stats.get("nonfinite_values", 1) != 0:
        hard.append("invalid_or_nonfinite_csv")
    if hard:
        return {"decision": "reject", "reason": hard}
    in_corridor = (
        1750 <= stats["detections_total"] <= 1885 and
        775 <= stats["empty_images"] <= 850 and
        710 <= stats["confidence_sum"] <= 765 and
        comp["precision_vs_beta0p05"] >= 0.975 and
        comp["recall_vs_beta0p05"] >= 0.970 and
        cpsr["zone"] in ["green", "low_orange"]
    )
    mechanism_ok = (poison_effect_ratio is not None and poison_effect_ratio >= 2.0 and outside_drift_ok is True)
    if in_corridor and mechanism_ok:
        return {"decision": "demander accord humain pour submit", "reason": "corridor+mechanism_ok; exact CSV/SHA approval required; no automatic submit"}
    if in_corridor:
        return {"decision": "rerun", "reason": "safe corridor but mechanism insufficient/unknown; refine objective or audit localization"}
    return {"decision": "reject", "reason": "outside submit-review corridor"}

def write_final_no_submit_report(candidate_audits=None):
    candidate_audits = candidate_audits or []
    lines = []
    lines.append(f"# {RUN_ID} — no-submit report")
    lines.append("")
    lines.append(f"Generated UTC: {datetime.now(timezone.utc).isoformat()}")
    lines.append("")
    lines.append("## Guardrails")
    lines.append("- no-submit only")
    lines.append("- auto_submit=false")
    lines.append("- submission_authorized=false")
    lines.append("- paid_gpu_authorized=false")
    lines.append("- no Kaggle submission code path")
    lines.append("")
    lines.append("## Planned final audit metrics")
    lines.append("detections_total, empty_images, confidence_sum, precision/recall vs beta0p05, CPSR, poison/outside effect ratio, frozen parameter drift.")
    lines.append("")
    if not candidate_audits:
        lines.append("## Decision")
        lines.append("reject")
        lines.append("")
        lines.append("Reason: preparation/preflight-only notebook; no training executed; no candidate CSV produced.")
    else:
        lines.append("## Candidate decisions")
        for audit in candidate_audits:
            decision = decide_no_submit(audit)
            lines.append(f"- {audit['candidate']}: {decision['decision']} — {decision['reason']}")
    path = OUTPUT_ROOT / "reports" / "controlled_unlearn_beta005_preserve_001_no_submit_report.md"
    path.write_text("\n".join(lines) + "\n", encoding="utf-8")
    return path

report_path = write_final_no_submit_report([])
print("Rapport no-submit préparatoire écrit:", report_path)
print("Décision préparatoire: reject (aucun entraînement exécuté, aucun CSV candidat produit).")


In [ ]:
# =============================
# 10. Manifest SHA256 des fichiers produits par ce notebook
# =============================
def build_manifest(root: Path):
    rows = []
    for p in sorted(root.rglob("*")):
        if p.is_file():
            rows.append({"path": str(p), "sha256": sha256_file(p), "bytes": p.stat().st_size})
    manifest = {"root": str(root), "files": rows, "created_at_utc": datetime.now(timezone.utc).isoformat()}
    write_json(root / "sha256_manifest.json", manifest)
    return manifest

manifest = build_manifest(OUTPUT_ROOT)
print(json.dumps(manifest, indent=2)[:4000])
